# 50 -- enumeration probe, step 2: dump hidden states

GPU required (rung 42 ep4's merged checkpoint). Resolves every manifest row from `01` to
its frame via the shared `frames_cache` (no copying -- RULES: the cache is the single
source, called from, never re-copied per experiment), then one forward pass per row with
`output_hidden_states=True`, keeping the last-prompt-token representation at a stride of
layers plus the three deepstack re-injection points (8, 16, 24).

`number` and `fo_class` rows are dumped in the SAME pass per pool (fit / final_read) --
the hidden state doesn't know or care which format the question is; it's read off the
prompt, before generation starts. `SMOKE=True` exercises the full pipeline on a handful of
rows per pool before committing to the full dump (~15.6k fit rows + 1.0k final-read rows).

In [ ]:
# --- parameters (RAW LITERALS ONLY -- papermill injects a new cell right after THIS one) --
MODEL_PATH = "/workspace/models/rung42_ep4_merged"
DATA_ROOT = "/workspace/orena-data"
FRAMES_CACHE = "/workspace/frames_cache"
OUT_DIR = "/workspace/repo/experiments/50-enumeration-probe/runs/50_hidden_v1"
SMOKE = True
SMOKE_ROWS = 12
LAYER_STRIDE = 2

In [ ]:
# --- bootstrap -----------------------------------------------------------------
import sys
from pathlib import Path
import pandas as pd

EXP = Path.cwd()
REPO = EXP
while REPO != REPO.parent and not (REPO / "src").is_dir():
    REPO = REPO.parent
if EXP.name != "50-enumeration-probe":
    EXP = REPO / "experiments" / "50-enumeration-probe"

if str(EXP / "_tools") not in sys.path:
    sys.path.insert(0, str(EXP / "_tools"))
for p in (REPO / "src", REPO / "vendor" / "orena-focus" / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import hidden_dump
import resolve_frames
from frame.config import BaselineConfig

print("repo:", REPO, "| exp:", EXP)

In [ ]:
# --- derived --------------------------------------------------------------------
DATA_ROOT = next(
    (Path(c) for c in (DATA_ROOT, str(REPO / "external_data" / "orena-data"))
     if (Path(c) / "heico" / "data" / "frame" / "test.parquet").exists()
     or (Path(c) / "heico" / "test.parquet").exists()),
    None)
assert DATA_ROOT is not None, "no frame test.parquet found (checked nested and flat layouts)"

cfg = BaselineConfig(data_root=DATA_ROOT)
cfg.frames_cache = Path(FRAMES_CACHE)
assert cfg.frames_cache.is_dir(), f"frames_cache not found: {cfg.frames_cache}"

OUT_DIR = Path(OUT_DIR)

In [ ]:
# --- 1. load the 01 manifests, resolve frames -------------------------------------
fit_number = pd.read_csv(EXP / "RESULTS_fit_number_v1.csv")
fit_foclass = pd.read_csv(EXP / "RESULTS_fit_foclass_v1.csv")
final_read = pd.read_csv(EXP / "RESULTS_final_read_v1.csv")

fit_all = pd.concat([fit_number, fit_foclass], ignore_index=True)
assert fit_all["qID"].is_unique, "number/fo_class fit rows should be disjoint by construction"

fit_all = resolve_frames.resolve_image_paths(fit_all, cfg)
final_read = resolve_frames.resolve_image_paths(final_read, cfg)
print(f"fit_all: {len(fit_all)} rows, frames resolved")
print(f"final_read: {len(final_read)} rows, frames resolved")

In [ ]:
# --- 2. smoke: fit pool -----------------------------------------------------------
smoke_cfg_fit = hidden_dump.Config(
    model_path=Path(MODEL_PATH), out_dir=OUT_DIR, layer_stride=LAYER_STRIDE,
    smoke=SMOKE, smoke_rows=SMOKE_ROWS, tag="fit",
)
hidden_dump.dump(smoke_cfg_fit, fit_all)

In [ ]:
# --- 3. smoke: final-read pool -----------------------------------------------------
smoke_cfg_final = hidden_dump.Config(
    model_path=Path(MODEL_PATH), out_dir=OUT_DIR, layer_stride=LAYER_STRIDE,
    smoke=SMOKE, smoke_rows=SMOKE_ROWS, tag="final_read",
)
hidden_dump.dump(smoke_cfg_final, final_read)

## Full dump

Only run the cells below after the smoke cells above have been inspected and look right
(shapes, layer list, no exceptions). Re-run this notebook with `SMOKE=False` via papermill
on the pod (`-p SMOKE False`) rather than editing the parameters cell in place -- keeps the
smoke-verified version in git history undisturbed.

In [ ]:
# --- 4. full dump: fit pool + final-read pool --------------------------------------
if not SMOKE:
    full_cfg_fit = hidden_dump.Config(
        model_path=Path(MODEL_PATH), out_dir=OUT_DIR, layer_stride=LAYER_STRIDE,
        smoke=False, tag="fit",
    )
    hidden_dump.dump(full_cfg_fit, fit_all)

    full_cfg_final = hidden_dump.Config(
        model_path=Path(MODEL_PATH), out_dir=OUT_DIR, layer_stride=LAYER_STRIDE,
        smoke=False, tag="final_read",
    )
    hidden_dump.dump(full_cfg_final, final_read)
else:
    print("SMOKE=True -- skipping the full dump, this cell is a no-op")